# 16 · Negation probe: does the model react to "without"? (supplement N1-B / T4)

**Question:** negation queries are the only group where stronger models did not help (final 0.853 vs 0.902 on the other queries). Is it the model (it ignores "without Y") or the labels (they do not penalise products that contain Y; see the label audit N1-A)? This notebook needs **no labels** for its main measures: it scores the same candidate list under versions of the query and checks the direction of the change.

For a query "X without Y" (`src/probes.py`):

| Version | Example |
|---|---|
| `raw` | the query as stored (used only to check that the checkpoint reproduces its saved scores) |
| `orig` | normalised query: "ipad case 8th generation without front cover" |
| `drop` | negated phrase removed: "ipad case 8th generation" |
| `flip` | exclusion turned into inclusion: "ipad case 8th generation with front cover" |
| `para` | same exclusion, other cue: "ipad case 8th generation no front cover" |

Each product is tagged with `term_status` (title + bullets): `has` Y, `negated` (the product itself says "no Y", "Y-free") or `absent`.

**Measures** (per model; mean over queries, bootstrap CI over queries):
- **dir_has:** share of `has` products with score(orig) < score(flip); a model that reads the exclusion scores them lower when Y is excluded. 0.5 = no reaction.
- **dir_negated:** share of `negated` products with score(orig) > score(flip).
- **sensitivity:** mean |score(orig) − score(drop)| / std of score(orig) in the list (scale-free, so MSE logits and ce4 expected gains compare).
- **invariance:** Kendall τ between the rankings for `orig` and `para` (should be close to 1).
- **ΔnDCG(orig − drop)** with the real labels: what the exclusion is worth under ESCI's labels.

**Models** (checkpoints): B3 MiniLM `title` MSE (`03_cross_encoder`), MiniLM `main` ce4 and the best distilled MiniLM (`14_distill`), DeBERTa-base ce4 s43 (`15_deberta_base_s43`). DeBERTa-large and the final system have no saved weights and are not probed.

**Query list:** chosen and checked by hand **before** any model is probed:
1. `MODE = 'prepare'` (CPU): writes `negation_probe_candidates.csv` (every val query with a parsed exclusion, plus its versions).
2. One member marks `keep` = 1 / 0 (0: not an exclusion, e.g. titles and idioms) and may correct `term`, `drop`, `flip`, `para` by hand; saves it as `negation_probe_queries.csv` and adds it to `wm-src` (any folder).
3. `MODE = 'smoke'` (first 20 kept queries, one model), then `MODE = 'full'`.
Only val queries are used (`SPLITS`); add `'test'` only if fewer than 150 val queries are kept, and record that choice before 29/11.

**Kaggle settings:** `prepare`: CPU. `smoke` / `full`: GPU **T4**, Internet off is fine (all models come from inputs).

**Inputs:** `wm-src` (supplement branch, with `src/probes.py` and, after step 2, `negation_probe_queries.csv`); **01_preprocess**; checkpoints: output of **03_cross_encoder** (`model_b3_minilm_title_mse_s42`), **14_distill** (`models/d4_minilm_main_*`), **15_deberta_base_s43** (`models/d2_deberta_main_ce4_s43`). Missing checkpoints are skipped with a message.

**Output** (`/kaggle/working/results/`): `negation_probe_candidates.csv` (prepare), `negation_probe_scores.parquet` (every version × product × model), `negation_probe.csv` (measures with CIs).

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import gc, glob, json, os

import numpy as np
import pandas as pd
from scipy.stats import kendalltau

from src.data import PREPROCESS_HINT, find, load_pairs, product_texts
from src.evaluation import query_metrics
from src.probes import term_status, variants

MODE = 'prepare'        # 'prepare' (CPU): candidate list | 'smoke': 20 queries, 1 model | 'full'
SPLITS = ['val']        # add 'test' only if fewer than 150 val queries are kept (decide before 29/11)
SEED = 42
VERSIONS = ['raw', 'orig', 'drop', 'flip', 'para']
# checkpoint folder name -> product text preset and max_len it was trained with
MODELS = {'b3_minilm_title_mse_s42': {'dir': 'model_b3_minilm_title_mse_s42', 'fields': 'title', 'max_len': 128},
          'd4_minilm_main_ce4_s42': {'dir': 'd4_minilm_main_ce4_s42', 'fields': 'main', 'max_len': 256},
          'd4_minilm_main_kd05_s42': {'dir': 'd4_minilm_main_kd05_s42', 'fields': 'main', 'max_len': 256},
          'd4_minilm_main_kd00_s42': {'dir': 'd4_minilm_main_kd00_s42', 'fields': 'main', 'max_len': 256},
          'd2_deberta_main_ce4_s43': {'dir': 'd2_deberta_main_ce4_s43', 'fields': 'main', 'max_len': 256}}
RES = Path('/kaggle/working/results')
RES.mkdir(parents=True, exist_ok=True)
assert MODE in ('prepare', 'smoke', 'full')
print('MODE', MODE, '| splits', SPLITS)

## 1. Candidate queries (`prepare`) or the hand-checked list

In [ ]:
pairs = pd.read_parquet(find('pairs.parquet', PREPROCESS_HINT),
                        columns=['example_id', 'query_id', 'query', 'product_id', 'esci_label', 'split'])
pairs = pairs[pairs['split'].isin(SPLITS)].reset_index(drop=True)
queries = pairs[['query_id', 'query', 'split']].drop_duplicates('query_id')

if MODE == 'prepare':
    rows = []
    for r in queries.itertuples():
        v = variants(r.query)
        if v is not None:
            rows.append({'query_id': r.query_id, 'split': r.split, 'query': r.query, 'cue': v['cue'], 'term': v['term'],
                         'orig': v['orig'], 'drop': v['drop'], 'flip': v['flip'], 'para': v['para'], 'keep': ''})
    cand = pd.DataFrame(rows).sort_values('query_id')
    cand.to_csv(RES / 'negation_probe_candidates.csv', index=False)
    print(f'{len(cand)} candidate queries of {len(queries)} ({cand["cue"].value_counts().to_dict()})')
    print('Next: mark keep = 1 / 0 by hand, save as negation_probe_queries.csv, add it to wm-src, run smoke.')
    display(cand.head(20))
else:
    hits = glob.glob('/kaggle/input/**/negation_probe_queries.csv', recursive=True)
    assert hits, 'negation_probe_queries.csv not found: run MODE = "prepare", check the list by hand, add it to wm-src'
    qlist = pd.read_csv(hits[0])
    qlist = qlist[(qlist['keep'] == 1) & qlist['split'].isin(SPLITS)].sort_values('query_id').reset_index(drop=True)
    if MODE == 'smoke':
        qlist = qlist.head(20)
    print(f'{len(qlist)} kept queries | cues {qlist["cue"].value_counts().to_dict()}')

## 2. Probe pairs: every version × every candidate of the query

One row per (query, version, product). `term_status` is computed once per (query, product) on the product title + bullets.

In [ ]:
if MODE != 'prepare':
    prod = pd.read_parquet(find('products_clean.parquet', PREPROCESS_HINT), columns=['product_id', 'title', 'bullets'])
    prod_text = (prod['title'].fillna('') + ' ' + prod['bullets'].fillna('')).set_axis(prod['product_id'])
    base = pairs.merge(qlist[['query_id', 'term', 'orig', 'drop', 'flip', 'para']], on='query_id')
    base['status'] = [term_status(prod_text.get(p, ''), t) for p, t in zip(base['product_id'], base['term'])]
    base = base.rename(columns={'query': 'raw'})
    probe = base.melt(id_vars=['example_id', 'query_id', 'product_id', 'esci_label', 'status'],
                      value_vars=VERSIONS, var_name='version', value_name='query').reset_index(drop=True)
    print(f'{len(base):,} (query, product) pairs -> {len(probe):,} probe rows | status',
          base['status'].value_counts().to_dict())
    assert base['status'].eq('has').any(), "no candidate contains the excluded term: check the query list"

## 3. Score with every checkpoint

Each checkpoint is loaded in fp16 and scores all probe rows with the product text it was trained on. Check: on `raw`, the DeBERTa s43 and MiniLM checkpoints must reproduce their saved val scores (if those score files are in the inputs).

In [ ]:
if MODE != 'prepare':
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from src.cross_encoder import DEVICE, predict
    assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4'

    def find_checkpoint(folder):
        hits = [Path(h).parent for h in glob.glob(f'/kaggle/input/**/{folder}/config.json', recursive=True)]
        return hits[0] if hits else None

    texts = {}
    scored = []
    names = list(MODELS)[:1] if MODE == 'smoke' else list(MODELS)
    for name in names:
        spec = MODELS[name]
        d = find_checkpoint(spec['dir'])
        if d is None:
            print(f'{name}: checkpoint {spec["dir"]} not in the inputs, skipped')
            continue
        if spec['fields'] not in texts:
            texts[spec['fields']] = product_texts(spec['fields'])
        df = probe.assign(text=probe['product_id'].map(texts[spec['fields']]))
        model = AutoModelForSequenceClassification.from_pretrained(d, torch_dtype=torch.float16).to(DEVICE)
        tok = AutoTokenizer.from_pretrained(d)
        s = predict(model, tok, df, spec['max_len'], batch_size=256, fp16=True)
        scored.append(probe[['example_id', 'query_id', 'version']].assign(model=name, score=s))
        saved = glob.glob(f'/kaggle/input/**/scores/{name}_val.parquet', recursive=True)
        if saved:
            ref = pd.read_parquet(saved[0], columns=['example_id', 'score'])
            raw = scored[-1][scored[-1]['version'] == 'raw'].merge(ref, on='example_id', suffixes=('', '_saved'))
            gap = float((raw['score'] - raw['score_saved']).abs().max())
            print(f'{name}: raw-query scores vs saved val scores, max |difference| {gap:.4f} over {len(raw):,} pairs')
            assert gap < 0.05, f'{name}: checkpoint or text preset does not reproduce the saved scores'
        print(f'{name}: scored {len(df):,} rows')
        del model
        gc.collect()
        torch.cuda.empty_cache()
    scores = pd.concat(scored, ignore_index=True)
    scores.to_parquet(RES / f"{'smoke_' if MODE == 'smoke' else ''}negation_probe_scores.parquet", index=False)

## 4. Measures

In [ ]:
def boot(x, n_boot=1000, seed=SEED):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if not len(x):
        return np.nan, [np.nan, np.nan], 0
    m = x[np.random.default_rng(seed).integers(0, len(x), (n_boot, len(x)))].mean(axis=1)
    return round(float(x.mean()), 4), [round(float(np.percentile(m, 2.5)), 4), round(float(np.percentile(m, 97.5)), 4)], len(x)


if MODE != 'prepare':
    out = []
    lab = base[['example_id', 'query_id', 'esci_label', 'status']]
    for name, g in scores.groupby('model'):
        w = g.pivot_table(index='example_id', columns='version', values='score').join(lab.set_index('example_id'))
        per_q = {}
        for qid, q in w.groupby('query_id'):
            has, neg = q[q['status'] == 'has'], q[q['status'] == 'negated']
            sd = q['orig'].std()
            per_q[qid] = {
                'dir_has': (has['orig'] < has['flip']).mean() if len(has) else np.nan,
                'dir_negated': (neg['orig'] > neg['flip']).mean() if len(neg) else np.nan,
                'sensitivity': (q['orig'] - q['drop']).abs().mean() / sd if len(q) > 1 and sd > 0 else np.nan,
                'invariance_tau': kendalltau(q['orig'], q['para'])[0] if len(q) > 2 else np.nan}
        per_q = pd.DataFrame.from_dict(per_q, orient='index')
        order = w.reset_index().sort_values(['query_id', 'example_id'])
        nd = {v: query_metrics(order, order[v].to_numpy()).set_index('query_id')['ndcg'] for v in ('orig', 'drop')}
        per_q['dndcg_orig_minus_drop'] = (nd['orig'] - nd['drop']).reindex(per_q.index)
        row = {'model': name}
        for c in per_q.columns:
            row[c], row[c + '_ci95'], row[c + '_n_queries'] = boot(per_q[c])
        out.append(row)
    result = pd.DataFrame(out)
    display(result)
    result.to_csv(RES / f"{'smoke_' if MODE == 'smoke' else ''}negation_probe.csv", index=False)
    print('Reading: dir_has / dir_negated near 0.5 = the model ignores the exclusion; well above 0.5 = it reacts. '
          'If it reacts but dNDCG(orig - drop) is about 0, the labels do not reward the exclusion (see N1-A).')